# Phase 05 — Baseline Evaluation Notebook

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Evaluate simple baselines before training complex models, so model improvement can be proven.

This notebook is the Phase 5 source of truth. It writes `reports/phase_05_baseline_evaluation.json` with regression baselines, ranking proxy metrics, slice diagnostics, error examples, and a training readiness decision.


## Contract boundary

Phase 5 evaluates transparent baselines only. It does not train complex neural models, calibrate production thresholds, create backend response copy, or change the public API contract.

Current available data is the legacy weak-label pair artifact. Because the Phase 4 balanced pair-generation implementation has not produced a new pair dataset yet, this notebook treats legacy metrics as prototype evidence and records missing Phase 4 columns as readiness blockers.


## Shared setup

### Purpose
Load the legacy pair dataset, Phase 2 baseline policy, Phase 3 normalization policy, and Phase 4 split/pair metadata expectations. Create deterministic validation/test splits by `profile_id` only when explicit split columns are missing.

### Required input
Repository root with `GAP_MODEL_TRAINING.md`, `legacy/artifacts/pairs.parquet`, `reports/phase_02_label_schema_baselines.json`, `reports/phase_03_normalization_feature_design.json`, and `reports/phase_04_pair_generation_splits.json`.

### Action
Build reviewable scalar features from allowed source columns only: skill overlap, text similarity, experience match, role match, score band, role-family slices, language slice placeholder, and pair-type placeholder. Never use profile ID, job ID, or validation labels as features.

### Expected output
A prepared evaluation dataframe with deterministic split names and feature columns, plus helper functions for regression metrics, ranking metrics, slices, and report writing.

### Verification
Setup must run with standard Python plus pandas, numpy, scipy, and scikit-learn. Generated output must be limited to `reports/phase_05_baseline_evaluation.json`.


In [7]:
from __future__ import annotations

import hashlib
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "GAP_MODEL_TRAINING.md").exists() and (candidate / "training").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd().resolve())
REPORTS = ROOT / "reports"
PAIRS_PARQUET = ROOT / "legacy" / "artifacts" / "pairs.parquet"
PHASE2_REPORT = REPORTS / "phase_02_label_schema_baselines.json"
PHASE3_REPORT = REPORTS / "phase_03_normalization_feature_design.json"
PHASE4_REPORT = REPORTS / "phase_04_pair_generation_splits.json"
PHASE5_REPORT = REPORTS / "phase_05_baseline_evaluation.json"
SPLIT_SEED = 20260601


def load_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {"missing": True, "path": str(path.relative_to(ROOT))}
    return json.loads(path.read_text())


def write_report(payload: dict[str, Any]) -> None:
    REPORTS.mkdir(parents=True, exist_ok=True)
    PHASE5_REPORT.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n")


def stable_unit_interval(value: object, seed: int = SPLIT_SEED) -> float:
    digest = hashlib.sha256(f"{seed}:{value}".encode("utf-8")).hexdigest()
    return int(digest[:12], 16) / float(16**12 - 1)


def assign_group_split(profile_id: object) -> str:
    u = stable_unit_interval(profile_id)
    if u < 0.70:
        return "train"
    if u < 0.85:
        return "validation"
    return "test"


def parse_skill_set(value: object) -> set[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return set()
    raw = str(value).lower()
    parts = re.split(r"[,;/|]+", raw)
    return {re.sub(r"\s+", " ", p.strip().replace(".", "")).strip() for p in parts if p.strip()}


def jaccard(a: set[str], b: set[str]) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def numeric_experience_band(value: object) -> str:
    try:
        v = float(value)
    except (TypeError, ValueError):
        return "unknown"
    if v <= 0:
        return "entry"
    if v <= 2:
        return "junior"
    if v <= 4:
        return "mid_level"
    if v <= 6:
        return "senior"
    return "lead_or_above"


def experience_match(profile_exp: object, job_exp: object) -> float:
    try:
        p = float(profile_exp)
        j = float(job_exp)
    except (TypeError, ValueError):
        return 0.0
    # Legacy pairs use numeric years/buckets. Matching is bounded and intentionally transparent.
    return float(max(0.0, 1.0 - min(abs(p - j), 5.0) / 5.0))


def extract_role(text: object) -> str:
    value = str(text or "").lower()
    match = re.search(r"(?:role|^)(?:\s+)([^.]+?)\.\s+skills", value)
    if not match:
        match = re.search(r"^([^.]*)\.\s+skills", value)
    role = match.group(1).strip() if match else "unknown"
    role = re.sub(r"[^a-z0-9 +#.-]+", " ", role)
    return re.sub(r"\s+", " ", role).strip() or "unknown"


def role_family(role: object) -> str:
    text = str(role or "").lower()
    families = [
        ("data", ["data", "machine learning", "ml", "ai", "analyst", "scientist"]),
        ("backend", ["backend", "back end", "api", "server"]),
        ("frontend", ["frontend", "front end", "web developer", "react", "vue"]),
        ("mobile", ["android", "ios", "mobile", "flutter"]),
        ("cloud_devops", ["cloud", "devops", "site reliability", "sre", "infrastructure"]),
        ("security", ["security", "soc", "siem", "cyber"]),
        ("design", ["designer", "ui", "ux"]),
        ("product", ["product", "project manager", "scrum"]),
        ("qa", ["quality", "qa", "tester", "test engineer"]),
    ]
    for family, needles in families:
        if any(needle in text for needle in needles):
            return family
    return "other"


def score_band(score: float) -> str:
    if score < 0.35:
        return "low"
    if score < 0.65:
        return "medium"
    return "high"


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def band_agreement(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    true_bands = [score_band(float(v)) for v in y_true]
    pred_bands = [score_band(float(v)) for v in y_pred]
    return float(np.mean([a == b for a, b in zip(true_bands, pred_bands)]))


def safe_spearman(y_true: np.ndarray, y_pred: np.ndarray) -> float | None:
    if len(np.unique(y_pred)) < 2 or len(np.unique(y_true)) < 2:
        return None
    value = spearmanr(y_true, y_pred).correlation
    if value is None or np.isnan(value):
        return None
    return float(value)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float | None]:
    clipped = np.clip(y_pred, 0.0, 1.0)
    return {
        "mae": float(mean_absolute_error(y_true, clipped)),
        "rmse": rmse(y_true, clipped),
        "r2": float(r2_score(y_true, clipped)) if len(np.unique(y_true)) > 1 else None,
        "spearman": safe_spearman(y_true, clipped),
        "score_band_agreement": band_agreement(y_true, clipped),
    }


def summarize_counts(frame: pd.DataFrame, columns: list[str]) -> dict[str, dict[str, int]]:
    result: dict[str, dict[str, int]] = {}
    for column in columns:
        if column in frame.columns:
            result[column] = {str(k): int(v) for k, v in frame[column].value_counts(dropna=False).sort_index().items()}
    return result

phase2 = load_json(PHASE2_REPORT)
phase3 = load_json(PHASE3_REPORT)
phase4 = load_json(PHASE4_REPORT)

required_pair_columns = [
    item["column"] for item in phase4.get("pair_metadata_schema", []) if item.get("required")
]
legacy_pairs = pd.read_parquet(PAIRS_PARQUET)
missing_phase4_columns = [c for c in required_pair_columns if c not in legacy_pairs.columns]

pairs = legacy_pairs.copy()
pairs["split"] = pairs["profile_id"].map(assign_group_split)
pairs["profile_skill_set"] = pairs["profile_skills"].map(parse_skill_set)
pairs["job_skill_set"] = pairs["job_skills"].map(parse_skill_set)
pairs["skill_overlap_score"] = [jaccard(a, b) for a, b in zip(pairs["profile_skill_set"], pairs["job_skill_set"])]
pairs["experience_match_score"] = [experience_match(a, b) for a, b in zip(pairs["profile_exp"], pairs["job_exp"])]
pairs["profile_role"] = pairs["profile_text"].map(extract_role)
pairs["job_role"] = pairs["job_text"].map(extract_role)
pairs["profile_role_family"] = pairs["profile_role"].map(role_family)
pairs["job_role_family"] = pairs["job_role"].map(role_family)
pairs["role_match_score"] = (pairs["profile_role_family"] == pairs["job_role_family"]).astype(float)
pairs["profile_experience_band"] = pairs["profile_exp"].map(numeric_experience_band)
pairs["job_experience_band"] = pairs["job_exp"].map(numeric_experience_band)
pairs["score_band"] = pairs["fit_score"].map(score_band)
pairs["pair_type"] = "legacy_weak_pair"
pairs["language"] = "UNKNOWN"
pairs["target"] = pairs["fit_score"].astype(float).clip(0.0, 1.0)

setup_summary = {
    "rows": int(len(pairs)),
    "source_pair_artifact": str(PAIRS_PARQUET.relative_to(ROOT)),
    "split_counts": summarize_counts(pairs, ["split"])["split"],
    "score_band_counts": summarize_counts(pairs, ["score_band"])["score_band"],
    "missing_phase4_required_columns": missing_phase4_columns,
    "prototype_limitations": [
        "Legacy pairs use weak-label fit_score instead of balanced Phase 4 job_fit_score.",
        "Explicit Phase 4 split, pair_type, language, semantic similarity, and requirement coverage columns are missing.",
        "High-fit labels are absent when using the Phase 2 high-band threshold.",
    ],
}
setup_summary


{'rows': 30000,
 'source_pair_artifact': 'legacy/artifacts/pairs.parquet',
 'split_counts': {'test': 4410, 'train': 21360, 'validation': 4230},
 'score_band_counts': {'low': 29443, 'medium': 557},
 'missing_phase4_required_columns': ['pair_id',
  'pair_type',
  'split',
  'score_band',
  'job_fit_score',
  'skill_overlap_score',
  'semantic_similarity_score',
  'experience_match_score',
  'role_match_score',
  'requirement_coverage_score',
  'language',
  'profile_role_family',
  'job_role_family',
  'profile_experience_band',
  'job_experience_band',
  'matched_skills',
  'missing_skills',
  'unknown_language',
  'unknown_experience',
  'empty_skills',
  'empty_text',
  'label_version',
  'feature_config_version',
  'source_dataset_hash'],
 'prototype_limitations': ['Legacy pairs use weak-label fit_score instead of balanced Phase 4 job_fit_score.',
  'Explicit Phase 4 split, pair_type, language, semantic similarity, and requirement coverage columns are missing.',
  'High-fit labels ar

## Step 5.1 — Regression baseline metrics

### Purpose
Document MAE, RMSE, R-squared, Spearman, and score-band agreement for constant and simple feature baselines.

### Required input
Use train/validation/test rows, weak-label target score, Phase 2 baseline catalog, and transparent scalar features available from source pair columns. Compute train-fitted values from the training split only.

### Action
Evaluate constant mean, constant median, skill-overlap-only, cosine-only TF-IDF similarity, and simple Ridge regression. Clip predictions to `0.0-1.0` because score labels are bounded.

### Expected output
A baseline metric table sorted by validation MAE with the best transparent baseline identified before complex model training.

### Verification
The best baseline must be selected from validation metrics only. Test metrics are reported for audit, not model selection. Negative or near-zero R-squared must remain visible.


In [8]:
train = pairs[pairs["split"] == "train"].copy()
validation = pairs[pairs["split"] == "validation"].copy()
test = pairs[pairs["split"] == "test"].copy()

y_train = train["target"].to_numpy()
y_val = validation["target"].to_numpy()
y_test = test["target"].to_numpy()

# Text similarity baseline: fit vocabulary only on training source text.
vectorizer = TfidfVectorizer(min_df=2, max_features=20000, ngram_range=(1, 2))
vectorizer.fit(pd.concat([train["profile_text"], train["job_text"]]).fillna(""))

def tfidf_pair_cosine(frame: pd.DataFrame) -> np.ndarray:
    profile_matrix = vectorizer.transform(frame["profile_text"].fillna(""))
    job_matrix = vectorizer.transform(frame["job_text"].fillna(""))
    values = cosine_similarity(profile_matrix, job_matrix).diagonal()
    return np.asarray(values, dtype=float)

train_cosine = tfidf_pair_cosine(train)
val_cosine = tfidf_pair_cosine(validation)
test_cosine = tfidf_pair_cosine(test)

train = train.assign(semantic_similarity_score=train_cosine)
validation = validation.assign(semantic_similarity_score=val_cosine)
test = test.assign(semantic_similarity_score=test_cosine)
pairs.loc[train.index, "semantic_similarity_score"] = train_cosine
pairs.loc[validation.index, "semantic_similarity_score"] = val_cosine
pairs.loc[test.index, "semantic_similarity_score"] = test_cosine

feature_columns = [
    "skill_overlap_score",
    "semantic_similarity_score",
    "experience_match_score",
    "role_match_score",
]

scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_columns])
X_val = scaler.transform(validation[feature_columns])
X_test = scaler.transform(test[feature_columns])
regression_model = Ridge(alpha=1.0, random_state=SPLIT_SEED)
regression_model.fit(X_train, y_train)

baseline_predictions = {
    "constant_mean": {
        "validation": np.full_like(y_val, float(np.mean(y_train)), dtype=float),
        "test": np.full_like(y_test, float(np.mean(y_train)), dtype=float),
    },
    "constant_median": {
        "validation": np.full_like(y_val, float(np.median(y_train)), dtype=float),
        "test": np.full_like(y_test, float(np.median(y_train)), dtype=float),
    },
    "skill_overlap_only": {
        "validation": validation["skill_overlap_score"].to_numpy(dtype=float),
        "test": test["skill_overlap_score"].to_numpy(dtype=float),
    },
    "cosine_only_tfidf": {
        "validation": val_cosine,
        "test": test_cosine,
    },
    "simple_ridge_regression": {
        "validation": regression_model.predict(X_val),
        "test": regression_model.predict(X_test),
    },
}

metric_rows: list[dict[str, Any]] = []
for baseline_name, prediction_sets in baseline_predictions.items():
    for split_name, y_true, y_pred in [
        ("validation", y_val, prediction_sets["validation"]),
        ("test", y_test, prediction_sets["test"]),
    ]:
        row = {"baseline": baseline_name, "split": split_name}
        row.update(regression_metrics(y_true, y_pred))
        metric_rows.append(row)

regression_metrics_table = pd.DataFrame(metric_rows).sort_values(["split", "mae", "rmse"]).reset_index(drop=True)
validation_metrics = regression_metrics_table[regression_metrics_table["split"] == "validation"].copy()
best_baseline_name = str(validation_metrics.sort_values(["mae", "rmse"]).iloc[0]["baseline"])
best_validation_prediction = np.clip(baseline_predictions[best_baseline_name]["validation"], 0.0, 1.0)
best_test_prediction = np.clip(baseline_predictions[best_baseline_name]["test"], 0.0, 1.0)

regression_coefficients = dict(zip(feature_columns, [float(v) for v in regression_model.coef_]))
regression_summary = {
    "best_baseline_by_validation_mae": best_baseline_name,
    "feature_columns": feature_columns,
    "simple_ridge_coefficients": regression_coefficients,
    "metrics": regression_metrics_table.to_dict(orient="records"),
}
regression_metrics_table


,baseline,split,mae,rmse,r2,spearman,score_band_agreement
0,simple_ridge_regression,test,0.016946,0.020595,0.960465,0.958141,0.987755
1,constant_median,test,0.085007,0.103610,-0.000645,NaN,0.980272
2,constant_mean,test,0.086133,0.103578,-0.000027,NaN,0.980272
3,cosine_only_tfidf,test,0.193208,0.218524,-3.451178,0.169813,0.980272
4,skill_overlap_only,test,0.196896,0.222504,-3.614800,0.063722,0.980045
5,simple_ridge_regression,validation,0.017261,0.021017,0.957909,0.964990,0.992435
6,constant_median,validation,0.082398,0.102837,-0.007742,NaN,0.984397
7,constant_mean,validation,0.083415,0.102608,-0.003260,NaN,0.984397
8,cosine_only_tfidf,validation,0.186717,0.212260,-3.293287,0.166002,0.984397
9,skill_overlap_only,validation,0.190651,0.216702,-3.474825,0.042638,0.984397


## Step 5.2 — Ranking baseline metrics

### Purpose
Document NDCG@5, NDCG@10, MAP@10, and candidate ordering examples when recommendation labels are available.

### Required input
Use validation rows grouped by `profile_id`. True recommendation labels are not available in the legacy artifact, so `fit_score` is used only as a weak proxy relevance signal for prototype diagnostics.

### Action
Compute NDCG with continuous weak relevance and MAP@10 with a documented weak binary proxy (`fit_score >= 0.35`). Compare all regression baselines as candidate orderers. Record ordering examples for the selected best regression baseline.

### Expected output
A ranking metric table and a small set of candidate ordering examples that demonstrate how each baseline sorts jobs for the same profile.

### Verification
Ranking metrics must be labeled as proxy metrics and must not be used for production readiness without backend candidate sets or human/application labels.


In [9]:
def dcg_at_k(relevance: np.ndarray, k: int) -> float:
    rel = np.asarray(relevance[:k], dtype=float)
    if rel.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, rel.size + 2))
    return float(np.sum(rel * discounts))


def ndcg_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int) -> float | None:
    if len(y_true) == 0:
        return None
    order = np.argsort(-y_score)
    ideal = np.argsort(-y_true)
    ideal_dcg = dcg_at_k(y_true[ideal], k)
    if ideal_dcg <= 0:
        return None
    return dcg_at_k(y_true[order], k) / ideal_dcg


def average_precision_at_k(y_true_binary: np.ndarray, y_score: np.ndarray, k: int) -> float | None:
    order = np.argsort(-y_score)[:k]
    hits = 0
    precisions: list[float] = []
    for rank, idx in enumerate(order, start=1):
        if y_true_binary[idx] > 0:
            hits += 1
            precisions.append(hits / rank)
    relevant = int(np.sum(y_true_binary))
    if relevant == 0:
        return None
    return float(np.sum(precisions) / min(relevant, k))


def grouped_ranking_metrics(frame: pd.DataFrame, prediction: np.ndarray, group_col: str = "profile_id") -> dict[str, Any]:
    eval_frame = frame[[group_col, "target"]].copy()
    eval_frame["prediction"] = prediction
    values: dict[str, list[float]] = {"ndcg_at_5": [], "ndcg_at_10": [], "map_at_10": []}
    skipped = 0
    groups = 0
    for _, group in eval_frame.groupby(group_col):
        if len(group) < 2:
            skipped += 1
            continue
        groups += 1
        y_true_group = group["target"].to_numpy(dtype=float)
        y_score_group = group["prediction"].to_numpy(dtype=float)
        binary_proxy = (y_true_group >= 0.35).astype(int)
        for metric, value in [
            ("ndcg_at_5", ndcg_at_k(y_true_group, y_score_group, 5)),
            ("ndcg_at_10", ndcg_at_k(y_true_group, y_score_group, 10)),
            ("map_at_10", average_precision_at_k(binary_proxy, y_score_group, 10)),
        ]:
            if value is not None and not np.isnan(value):
                values[metric].append(float(value))
    return {
        "evaluated_profile_groups": int(groups),
        "skipped_profile_groups": int(skipped),
        "ndcg_at_5": float(np.mean(values["ndcg_at_5"])) if values["ndcg_at_5"] else None,
        "ndcg_at_10": float(np.mean(values["ndcg_at_10"])) if values["ndcg_at_10"] else None,
        "map_at_10": float(np.mean(values["map_at_10"])) if values["map_at_10"] else None,
    }

ranking_rows: list[dict[str, Any]] = []
for baseline_name, prediction_sets in baseline_predictions.items():
    row = {"baseline": baseline_name, "split": "validation", "relevance_source": "legacy_fit_score_weak_proxy"}
    row.update(grouped_ranking_metrics(validation, np.clip(prediction_sets["validation"], 0.0, 1.0)))
    ranking_rows.append(row)

ranking_metrics_table = pd.DataFrame(ranking_rows).sort_values(["ndcg_at_10", "ndcg_at_5"], ascending=False, na_position="last").reset_index(drop=True)

example_profiles = (
    validation.assign(best_prediction=best_validation_prediction)
    .groupby("profile_id")
    .filter(lambda g: len(g) >= 5)
    .groupby("profile_id")["target"]
    .std()
    .sort_values(ascending=False)
    .head(3)
    .index
)
ordering_examples: list[dict[str, Any]] = []
example_frame = validation.assign(best_prediction=best_validation_prediction)
for profile_id in example_profiles:
    group = example_frame[example_frame["profile_id"] == profile_id].copy()
    top_pred = group.sort_values("best_prediction", ascending=False).head(5)
    top_true = group.sort_values("target", ascending=False).head(5)
    ordering_examples.append({
        "profile_id": str(profile_id),
        "baseline": best_baseline_name,
        "top_by_prediction": top_pred[["job_id", "job_role", "target", "best_prediction"]].to_dict(orient="records"),
        "top_by_label": top_true[["job_id", "job_role", "target", "best_prediction"]].to_dict(orient="records"),
    })

ranking_summary = {
    "metrics_are_proxy_only": True,
    "proxy_relevance": "legacy fit_score; MAP@10 binary proxy uses fit_score >= 0.35 because high-fit labels are absent",
    "metrics": ranking_metrics_table.to_dict(orient="records"),
    "ordering_examples": ordering_examples,
}
ranking_metrics_table


,baseline,split,relevance_source,evaluated_profile_groups,skipped_profile_groups,ndcg_at_5,ndcg_at_10,map_at_10
0,simple_ridge_regression,validation,legacy_fit_score_weak_proxy,423,0,0.999576,0.999764,1.000000
1,cosine_only_tfidf,validation,legacy_fit_score_weak_proxy,423,0,0.790396,0.910589,0.745056
2,skill_overlap_only,validation,legacy_fit_score_weak_proxy,423,0,0.748324,0.895928,0.923729
3,constant_mean,validation,legacy_fit_score_weak_proxy,423,0,0.727366,0.882377,0.339851
4,constant_median,validation,legacy_fit_score_weak_proxy,423,0,0.727366,0.882377,0.339851


## Step 5.3 — Slice analysis

### Purpose
Report metrics by role, language, experience band, pair type, and score band to expose weak spots.

### Required input
Use validation rows, the selected best baseline, derived role-family features, derived experience bands, placeholder language, placeholder pair type, and score bands from the weak label.

### Action
Compute MAE, RMSE, R-squared where possible, score-band agreement, and row counts for each slice value. Rank weak slices by MAE and call out missing or low-coverage slices.

### Expected output
A slice metrics table and weak-slice summary for role, language, experience, pair type, and score band.

### Verification
Slices with missing explicit Phase 4 fields must be marked as derived or placeholder evidence. No slice should hide UNKNOWN language or absent high-band examples.


In [10]:
slice_eval = validation.copy()
slice_eval["prediction"] = best_validation_prediction

slice_columns = [
    "profile_role_family",
    "job_role_family",
    "language",
    "profile_experience_band",
    "job_experience_band",
    "pair_type",
    "score_band",
]

slice_rows: list[dict[str, Any]] = []
for column in slice_columns:
    for value, group in slice_eval.groupby(column, dropna=False):
        y_true_group = group["target"].to_numpy(dtype=float)
        y_pred_group = group["prediction"].to_numpy(dtype=float)
        metrics = regression_metrics(y_true_group, y_pred_group)
        slice_rows.append({
            "slice_column": column,
            "slice_value": str(value),
            "rows": int(len(group)),
            **metrics,
        })

slice_metrics_table = pd.DataFrame(slice_rows).sort_values(["mae", "rows"], ascending=[False, False]).reset_index(drop=True)
weak_slices = slice_metrics_table[slice_metrics_table["rows"] >= 30].head(12).to_dict(orient="records")

slice_summary = {
    "baseline": best_baseline_name,
    "slice_evidence_notes": {
        "language": "Placeholder UNKNOWN because legacy pair artifact has no audited language column.",
        "pair_type": "Placeholder legacy_weak_pair because Phase 4 pair taxonomy has not been materialized.",
        "role_family": "Derived from source role/title text for diagnostics only.",
        "experience_band": "Derived from legacy numeric profile_exp/job_exp columns.",
    },
    "metrics": slice_metrics_table.to_dict(orient="records"),
    "weak_slices_min_30_rows": weak_slices,
}
slice_metrics_table.head(20)


,slice_column,slice_value,rows,mae,rmse,r2,spearman,score_band_agreement
0,job_experience_band,mid_level,804,0.034646,0.036661,0.523219,0.911794,1.000000
1,score_band,medium,66,0.030418,0.031914,-0.149592,0.907959,0.515152
2,job_experience_band,senior,430,0.024841,0.025622,-1.396949,0.274426,1.000000
3,job_role_family,product,172,0.022452,0.024989,0.935073,0.953175,1.000000
4,job_role_family,backend,228,0.020448,0.024715,0.944853,0.977336,0.982456
5,job_role_family,cloud_devops,335,0.019508,0.023831,0.945764,0.971903,0.994030
6,job_role_family,mobile,71,0.019447,0.022889,0.944110,0.971899,0.985915
7,profile_experience_band,entry,1210,0.019233,0.023442,0.910626,0.916086,0.999174
8,job_role_family,security,164,0.018498,0.022059,0.960613,0.961917,1.000000
9,profile_role_family,backend,680,0.018124,0.021991,0.957325,0.969300,0.989706


## Step 5.4 — Error inspection

### Purpose
Describe representative false-high and false-low examples with probable root causes.

### Required input
Use validation rows, the selected best baseline prediction, source role/skill/experience fields, and residuals.

### Action
Identify false-high examples where prediction is much higher than weak label and false-low examples where prediction is much lower than weak label. Attach probable root causes using only observable source evidence.

### Expected output
Representative error examples that reviewers can inspect without reading model internals.

### Verification
Examples must not include backend-owned product copy, user PII beyond legacy profile/job IDs, or inferred claims that are unsupported by source fields.


In [11]:
error_eval = validation.copy()
error_eval["prediction"] = best_validation_prediction
error_eval["residual"] = error_eval["prediction"] - error_eval["target"]
error_eval["abs_error"] = error_eval["residual"].abs()


def probable_root_cause(row: pd.Series) -> list[str]:
    causes: list[str] = []
    if row["skill_overlap_score"] == 0:
        causes.append("no exact normalized skill overlap")
    elif row["skill_overlap_score"] < 0.15:
        causes.append("low exact skill overlap")
    if row["role_match_score"] == 0:
        causes.append("profile/job role-family mismatch")
    if row["experience_match_score"] < 0.6:
        causes.append("experience gap")
    if row["semantic_similarity_score"] < 0.1:
        causes.append("low text similarity")
    if row["score_band"] == "low" and row["prediction"] >= 0.35:
        causes.append("baseline overestimates low-band weak label")
    if row["score_band"] == "medium" and row["prediction"] < 0.35:
        causes.append("baseline underestimates medium-band weak label")
    return causes or ["requires manual review"]


def example_records(frame: pd.DataFrame, n: int = 8) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    for _, row in frame.head(n).iterrows():
        records.append({
            "profile_id": str(row["profile_id"]),
            "job_id": str(row["job_id"]),
            "profile_role": row["profile_role"],
            "job_role": row["job_role"],
            "target": float(row["target"]),
            "prediction": float(row["prediction"]),
            "residual": float(row["residual"]),
            "skill_overlap_score": float(row["skill_overlap_score"]),
            "semantic_similarity_score": float(row["semantic_similarity_score"]),
            "experience_match_score": float(row["experience_match_score"]),
            "role_match_score": float(row["role_match_score"]),
            "probable_root_causes": probable_root_cause(row),
        })
    return records

false_high_examples = example_records(error_eval.sort_values("residual", ascending=False))
false_low_examples = example_records(error_eval.sort_values("residual", ascending=True))
high_abs_error_examples = example_records(error_eval.sort_values("abs_error", ascending=False))

error_summary = {
    "baseline": best_baseline_name,
    "false_high_examples": false_high_examples,
    "false_low_examples": false_low_examples,
    "highest_absolute_error_examples": high_abs_error_examples,
    "error_policy": "Examples are diagnostic only because labels are weak and high-fit labels are absent.",
}
error_summary["false_high_examples"][:3]


[{'profile_id': '43650',
  'job_id': 'e84fda68-400a-4c9f-a7cc-74f4b9555feb',
  'profile_role': 'cloud engineer',
  'job_role': 'devops engineer',
  'target': 0.0737,
  'prediction': 0.15143899226593793,
  'residual': 0.07773899226593793,
  'skill_overlap_score': 0.13333333333333333,
  'semantic_similarity_score': 0.09168574792819374,
  'experience_match_score': 0.4,
  'role_match_score': 1.0,
  'probable_root_causes': ['low exact skill overlap',
   'experience gap',
   'low text similarity']},
 {'profile_id': '5865',
  'job_id': '9922ee42-0c23-4921-b32f-ce4e196ebacf',
  'profile_role': 'cloud engineer',
  'job_role': 'software engineering manager',
  'target': 0.1233,
  'prediction': 0.19519611777127396,
  'residual': 0.07189611777127396,
  'skill_overlap_score': 0.21428571428571427,
  'semantic_similarity_score': 0.1100461926961232,
  'experience_match_score': 0.4,
  'role_match_score': 0.0,
  'probable_root_causes': ['profile/job role-family mismatch',
   'experience gap']},
 {'profi

## Step 5.5 — Training readiness gate

### Purpose
State whether labels and splits are strong enough to start model training or whether pair generation must be fixed first.

### Required input
Use Phase 5 metric evidence, Phase 2 baseline acceptance policy, Phase 4 pair metadata expectations, observed split/score distribution, ranking proxy limitations, and error/slice findings.

### Action
Compare the available evidence against the minimum requirements for model training. Record a go/no-go decision with blockers and required next actions.

### Expected output
An explicit training readiness decision and a machine-readable Phase 5 report.

### Verification
The decision must be evidence-based. If labels, high-fit coverage, explicit splits, or ranking labels are missing, the gate must not pass silently.


In [12]:
best_validation_row = validation_metrics.sort_values(["mae", "rmse"]).iloc[0].to_dict()
constant_mean_val = validation_metrics[validation_metrics["baseline"] == "constant_mean"].iloc[0].to_dict()
relative_mae_improvement_vs_mean = (
    (float(constant_mean_val["mae"]) - float(best_validation_row["mae"])) / float(constant_mean_val["mae"])
    if float(constant_mean_val["mae"]) > 0 else None
)

validation_score_bands = set(validation["score_band"].unique())
test_score_bands = set(test["score_band"].unique())
high_fit_validation_count = int((validation["score_band"] == "high").sum())
high_fit_test_count = int((test["score_band"] == "high").sum())

readiness_blockers = []
if missing_phase4_columns:
    readiness_blockers.append("Balanced Phase 4 pair metadata columns are missing from the available pair artifact.")
if high_fit_validation_count == 0 or high_fit_test_count == 0:
    readiness_blockers.append("Validation/test splits have no high-fit examples under the Phase 2 high-band threshold.")
if best_validation_row.get("r2") is None or float(best_validation_row["r2"]) <= 0:
    readiness_blockers.append("Best validation baseline does not show positive R-squared on the current weak-label target.")
if not relative_mae_improvement_vs_mean or relative_mae_improvement_vs_mean < 0.15:
    readiness_blockers.append("Best baseline does not improve MAE by at least 15% versus constant mean.")
if set(["low", "medium", "high"]) - validation_score_bands:
    readiness_blockers.append("Validation split is missing at least one score band.")
if set(["low", "medium", "high"]) - test_score_bands:
    readiness_blockers.append("Test split is missing at least one score band.")
readiness_blockers.append("True recommendation relevance labels and backend candidate sets are not available; ranking metrics are proxy-only.")
readiness_blockers.append("Human-labeled validation data is not available; legacy fit_score remains weak-label prototype evidence.")

training_readiness = {
    "decision": "NO_GO_FIX_PAIR_GENERATION_AND_LABELS_FIRST",
    "rationale": "Baseline evaluation is complete for the available legacy weak-label artifact, but evidence is not strong enough to start complex model training.",
    "best_baseline": best_baseline_name,
    "best_validation_metrics": best_validation_row,
    "relative_mae_improvement_vs_constant_mean": relative_mae_improvement_vs_mean,
    "validation_score_bands": sorted(validation_score_bands),
    "test_score_bands": sorted(test_score_bands),
    "high_fit_validation_count": high_fit_validation_count,
    "high_fit_test_count": high_fit_test_count,
    "blockers": readiness_blockers,
    "required_next_actions": [
        "Materialize Phase 4 balanced pair dataset with required metadata columns.",
        "Add high-fit positive pairs to validation and test splits.",
        "Add audited language, pair_type, role-family, and experience-band slices to the pair artifact.",
        "Add human/manual validation sample before claiming production readiness.",
        "Add backend candidate-set ranking labels before using NDCG/MAP as production recommendation evidence.",
    ],
}

phase5_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "inputs": {
        "phase_scope": "Phase 5 — Baseline Evaluation Notebook",
        "pairs_parquet": str(PAIRS_PARQUET.relative_to(ROOT)),
        "phase2_report": str(PHASE2_REPORT.relative_to(ROOT)),
        "phase3_report": str(PHASE3_REPORT.relative_to(ROOT)),
        "phase4_report": str(PHASE4_REPORT.relative_to(ROOT)),
    },
    "setup_summary": setup_summary,
    "acceptance": {
        "best_baseline_is_known_before_model_training": bool(best_baseline_name),
        "weak_slices_are_documented": bool(weak_slices),
        "training_readiness_decision_is_explicit": bool(training_readiness["decision"]),
    },
    "regression_baselines": regression_summary,
    "ranking_baselines": ranking_summary,
    "slice_analysis": slice_summary,
    "error_inspection": error_summary,
    "training_readiness_gate": training_readiness,
    "blocked_until_later_phases": [
        "Phase 6 complex job-fit training must wait for balanced pairs, high-fit coverage, and stronger labels.",
        "Phase 9 recommendation ranking must wait for backend candidate-set labels or realistic candidate sets.",
        "Phase 10 calibration must wait for stable label distribution and model outputs.",
    ],
}

write_report(phase5_report)
training_readiness


{'decision': 'NO_GO_FIX_PAIR_GENERATION_AND_LABELS_FIRST',
 'rationale': 'Baseline evaluation is complete for the available legacy weak-label artifact, but evidence is not strong enough to start complex model training.',
 'best_baseline': 'simple_ridge_regression',
 'best_validation_metrics': {'baseline': 'simple_ridge_regression',
  'split': 'validation',
  'mae': 0.01726078006981349,
  'rmse': 0.021016879423409764,
  'r2': 0.9579090970833964,
  'spearman': 0.9649904957896672,
  'score_band_agreement': 0.992434988179669},
 'relative_mae_improvement_vs_constant_mean': 0.7930732300833666,
 'validation_score_bands': ['low', 'medium'],
 'test_score_bands': ['low', 'medium'],
 'high_fit_validation_count': 0,
 'high_fit_test_count': 0,
 'blockers': ['Balanced Phase 4 pair metadata columns are missing from the available pair artifact.',
  'Validation/test splits have no high-fit examples under the Phase 2 high-band threshold.',
  'Validation split is missing at least one score band.',
  'Tes

## Acceptance criteria

- [x] Best baseline is known before model training.
- [x] Weak slices are documented.
- [x] Training readiness decision is explicit.


## Phase notes

- The best available baseline is selected from validation MAE on the legacy weak-label artifact.
- Training readiness is **NO-GO** because balanced Phase 4 pair rows, high-fit validation/test examples, human validation labels, and true ranking relevance labels are missing.
- Phase 6 must not start complex model training until pair generation and label quality blockers are fixed.
